# Computer Exercise 13.8 — Problem 1

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.)
> **단원**: 13.8 Minimization — *Nonlinear Least Squares: the Gauss–Newton Method*
> **풀이 일자**: 2026-06-27 · **언어**: 본문 한국어 / 그래프 라벨 영문


## 1. 문제 (원문)

> **1.** Given data $\{(t_i,y_i)\}_{i=1}^{m}$ and a model $\varphi(t;\mathbf{x})$ that depends
> nonlinearly on parameters $\mathbf{x}\in\mathbb{R}^n$, the *nonlinear least squares* problem is
> to minimize $f(\mathbf{x})=\tfrac12\sum_{i=1}^m r_i(\mathbf{x})^2$, where
> $r_i(\mathbf{x})=\varphi(t_i;\mathbf{x})-y_i$.
> Implement the **Gauss–Newton method**, in which the Hessian $\nabla^2 f$ is approximated by
> $J^\top J$ (dropping the second-order term). Fit the exponential model
> $\varphi(t;a,b)=a\,e^{bt}$ to data. Compare Gauss–Newton with the **full Newton** iteration
> (which keeps the curvature term $S=\sum_i r_i\nabla^2 r_i$), and observe how the convergence
> behaviour changes between a **small-residual** and a **large-residual** problem.

### 한국어 풀이용 정리
- 모형 $\varphi(t;a,b)=a\,e^{bt}$ 로 데이터를 적합(fitting)한다.
- 목적함수 $f(\mathbf{x})=\tfrac12\lVert \mathbf{r}(\mathbf{x})\rVert^2$ 를 최소화.
- **Gauss–Newton**: 정규방정식 $J^\top J\,\boldsymbol{\delta}=-J^\top\mathbf{r}$ 로 스텝 계산.
- **full Newton** 과 비교, 그리고 **작은 잔차 vs 큰 잔차** 에서 수렴 속도 차이를 확인.


## 2. 수학적 배경

### 2.1 비선형 최소제곱
잔차벡터 $\mathbf{r}(\mathbf{x})=(r_1,\dots,r_m)^\top$, $r_i(\mathbf{x})=\varphi(t_i;\mathbf{x})-y_i$ 에 대해
$$
f(\mathbf{x})=\tfrac12\,\mathbf{r}(\mathbf{x})^\top\mathbf{r}(\mathbf{x}),\qquad
\nabla f=J^\top\mathbf{r},\qquad
\nabla^2 f=J^\top J+\underbrace{\sum_{i=1}^m r_i\,\nabla^2 r_i}_{=\,S},
$$
여기서 $J\in\mathbb{R}^{m\times n}$ 는 야코비안 $J_{ij}=\partial r_i/\partial x_j$.

### 2.2 Gauss–Newton vs full Newton
**full Newton** 스텝: $(J^\top J+S)\,\boldsymbol{\delta}=-J^\top\mathbf{r}$.
**Gauss–Newton** 은 곡률항 $S$ 를 버린다:
$$
\boxed{\,J^\top J\,\boldsymbol{\delta}_{\mathrm{GN}}=-J^\top\mathbf{r}\,}
$$
$S=\sum_i r_i\nabla^2 r_i$ 는 잔차 $r_i$ 에 비례하므로,

* **작은 잔차**($r_i\approx0$) → $S\approx0$ → GN $\approx$ Newton → **거의 2차 수렴**.
* **큰 잔차**(모형 불일치/강한 잡음) → $S$ 가 무시 못 할 만큼 커짐 → GN 은 선형 수렴으로 **둔화**.

### 2.3 이 문제의 야코비안 ($\varphi=a\,e^{bt}$)
$$
\frac{\partial r_i}{\partial a}=e^{bt_i},\qquad
\frac{\partial r_i}{\partial b}=a\,t_i\,e^{bt_i}.
$$


## 3. 풀이 흐름

1. **데이터 생성**: 참 파라미터 $(a^\*,b^\*)=(2.5,-0.6)$ 로 $y=a^\*e^{b^\*t}$ 에 *작은* 잡음을 더해 small-residual 데이터를 만든다.
2. **모형/잔차/야코비안** 함수 구현.
3. **Gauss–Newton 루프**: $J^\top J\,\boldsymbol\delta=-J^\top\mathbf r$ 를 풀어 $\mathbf x\leftarrow\mathbf x+\boldsymbol\delta$, $\lVert J^\top\mathbf r\rVert<\text{tol}$ 에서 종료.
4. **full Newton 루프**: 곡률항 $S$ 를 포함해 같은 데이터에 적용.
5. **반복 표** 출력: $k,a,b,\lVert\mathbf r\rVert,\lVert J^\top\mathbf r\rVert,\text{cost}$.
6. **큰 잔차** 데이터(강한 잡음+편향)를 만들어 GN 반복수 변화를 관찰.
7. **시각화**: 적합 곡선, 그리고 small-residual 에서 GN vs Newton 의 수렴.


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

a_true, b_true = 2.5, -0.6
t = np.linspace(0.0, 4.0, 25)
y_clean = a_true * np.exp(b_true * t)
y = y_clean + 0.01 * rng.standard_normal(t.size)   # 작은 잔차

def model(x, tt):
    a, b = x
    return a * np.exp(b * tt)

def residual(x):
    return model(x, t) - y

def jac(x):
    a, b = x
    e = np.exp(b * t)
    J = np.empty((t.size, 2))
    J[:, 0] = e
    J[:, 1] = a * t * e
    return J

print("m =", t.size, "data points;  true (a,b) =", (a_true, b_true))


m = 25 data points;  true (a,b) = (2.5, -0.6)


In [2]:
def gauss_newton(x0, tol=1e-10, maxit=100):
    x = np.array(x0, float); hist = []
    for k in range(maxit):
        r = residual(x); J = jac(x); g = J.T @ r
        hist.append((k, x[0], x[1], np.sqrt(r @ r), np.linalg.norm(g), 0.5 * r @ r))
        if np.linalg.norm(g) < tol:
            break
        x = x + np.linalg.solve(J.T @ J, -g)
    cols = ["k", "a", "b", "||r||", "||grad||", "cost"]
    return x, pd.DataFrame(hist, columns=cols)

x_gn, hist_gn = gauss_newton([1.0, -0.1])
print("Gauss-Newton  ->  (a,b) =", x_gn, "   iters =", len(hist_gn) - 1)
x_gn


Gauss-Newton  ->  (a,b) = [ 2.50295571 -0.60200792]    iters = 6


array([ 2.50295571, -0.60200792])

In [3]:
# full Newton : H = J^T J + S
def full_newton(x0, tol=1e-10, maxit=100):
    x = np.array(x0, float); hist = []
    for k in range(maxit):
        a, b = x; e = np.exp(b * t)
        r = a * e - y
        J = np.column_stack([e, a * t * e]); g = J.T @ r
        S = np.zeros((2, 2))
        S[0, 1] = S[1, 0] = np.sum(r * t * e)
        S[1, 1] = np.sum(r * a * t * t * e)
        H = J.T @ J + S
        hist.append((k, np.sqrt(r @ r), np.linalg.norm(g)))
        if np.linalg.norm(g) < tol:
            break
        x = x + np.linalg.solve(H, -g)
    return x, pd.DataFrame(hist, columns=["k", "||r||", "||grad||"])

x_fn, hist_fn = full_newton([1.0, -0.1])
print("full Newton   ->  (a,b) =", x_fn, "   iters =", len(hist_fn) - 1)
x_fn


full Newton   ->  (a,b) = [ 2.50295571 -0.60200792]    iters = 5


array([ 2.50295571, -0.60200792])

In [4]:
pd.set_option("display.float_format", lambda v: f"{v:.6e}")
hist_gn


,k,a,b,||r||,||grad||,cost
0,0,1.000000e+00,-1.000000e-01,2.925692e+00,7.159330e+00,4.279837e+00
1,1,2.170960e+00,-6.405571e-01,9.249884e-01,4.192555e+00,4.278018e-01
2,2,2.501684e+00,-5.939766e-01,6.335825e-02,3.037395e-01,2.007134e-03
3,3,2.502866e+00,-6.019137e-01,4.195881e-02,2.665597e-03,8.802711e-04
4,4,2.502955e+00,-6.020077e-01,4.195633e-02,3.593658e-06,8.801670e-04
5,5,2.502956e+00,-6.020079e-01,4.195633e-02,6.497069e-09,8.801670e-04
6,6,2.502956e+00,-6.020079e-01,4.195633e-02,1.193955e-11,8.801670e-04


In [5]:
# 큰 잔차(large residual) 문제
y_big = y_clean + 0.6 * rng.standard_normal(t.size) + 0.7

def gn_generic(resfun, x0, tol=1e-10, maxit=500):
    x = np.array(x0, float); hist = []
    for k in range(maxit):
        a, b = x; e = np.exp(b * t)
        r = resfun(x)
        J = np.column_stack([e, a * t * e]); g = J.T @ r
        hist.append((k, np.linalg.norm(g), 0.5 * r @ r))
        if np.linalg.norm(g) < tol:
            break
        try:
            d = np.linalg.solve(J.T @ J, -g)
        except np.linalg.LinAlgError:
            break
        x = x + d
        if not np.all(np.isfinite(x)):
            break
    return x, pd.DataFrame(hist, columns=["k", "||grad||", "cost"])

x_big, hist_big = gn_generic(lambda x: model(x, t) - y_big, [1.0, -0.1])
small_iters = len(hist_gn) - 1
big_iters = len(hist_big) - 1
print(f"small-residual GN iters = {small_iters}")
print(f"large-residual GN iters = {big_iters}   (final cost={hist_big['cost'].iloc[-1]:.3e})")
pd.DataFrame({"problem": ["small residual", "large residual"],
             "GN iters": [small_iters, big_iters],
             "final cost": [hist_gn["cost"].iloc[-1], hist_big["cost"].iloc[-1]]})


small-residual GN iters = 6
large-residual GN iters = 13   (final cost=2.825e+00)


,problem,GN iters,final cost
0,small residual,6,8.801670e-04
1,large residual,13,2.824877e+00


In [6]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
tt = np.linspace(0, 4, 200)
ax[0].plot(t, y, "o", ms=4, color="#444", label="data")
ax[0].plot(tt, model(x_gn, tt), "-", color="#c0392b", lw=2, label="Gauss-Newton fit")
ax[0].set_xlabel("t"); ax[0].set_ylabel("y")
ax[0].set_title(r"Exponential fit  $y=a\,e^{bt}$"); ax[0].legend()

ax[1].semilogy(hist_gn["k"], hist_gn["||grad||"], "o-", ms=5, color="#c0392b", label="Gauss-Newton")
ax[1].semilogy(hist_fn["k"], hist_fn["||grad||"], "s--", ms=5, color="#2c3e50", label="full Newton")
ax[1].set_xlabel("iteration k"); ax[1].set_ylabel(r"$\|J^T r\|$")
ax[1].set_title("Convergence (small residual)"); ax[1].legend()
plt.tight_layout(); plt.show()
print("done")


done


## 4. 결과 해석

1. **작은 잔차에서 GN $\approx$ Newton**: 모형이 데이터를 거의 완벽히 설명하므로 곡률항 $S=\sum_i r_i\nabla^2 r_i$ 가 $\mathbf 0$ 에 가깝다. 두 방법 모두 **소수 회 반복**(GN 6회, Newton 5회)으로 $\lVert J^\top\mathbf r\rVert$ 이 거의 2차로 0 에 수렴한다.
2. **full Newton 이 한 발 빠르다**: 곡률항을 살리므로 마지막 자리수에서 더 가파르지만, 야코비안만 쓰는 GN 의 단순함 대비 이득은 작다.
3. **큰 잔차에서 둔화**: 잡음+편향으로 잔차가 커지면 $S$ 를 무시할 수 없어 반복수가 6→13 회로 늘고 수렴이 **선형**에 가까워진다.
4. **핵심 trade-off**: GN 은 2계 도함수 없이 야코비안만으로 헤시안을 근사 → 값싸고 small-residual 에서 거의 2차. 단, large-residual 에서 근사 오차가 드러난다.

> **결론**: Gauss–Newton 은 $\nabla^2 f\approx J^\top J$ 근사로 2계 도함수를 피하며, **잔차가 작을수록 Newton 에 수렴(거의 2차)** 하지만 **잔차가 크면 곡률항 누락으로 둔화**한다.

**다음 문제로의 연결** — 큰 잔차/나쁜 초기값에서 $J^\top J$ 가 특이하거나 스텝이 폭주하는 것을 막으려면 감쇠(damping)가 필요하다. 다음 노트북(§13.8-2)에서 **Levenberg–Marquardt** 로 보완한다.
